# **MultiPDFChat - Document Interaction Simplified**

## **Step-1:- Install required libraries**

In [3]:
%pip install PyPDF2 langchain langchain_google_genai google-generativeai faiss-cpu langchain-text-splitters langchain-community

INFO: pip is looking at multiple versions of google-api-core to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of google-api-core to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of google-api-core[grpc] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of google-api-core[grpc] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of googleapis-common-protos to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of httpcore2 to determine which version is compatible with other requirements. This could take a while.
  Using cached httpx2-2.13.1-py3-none-any.whl.metadata (9.8 k

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
python-jobspy 1.1.82 requires regex<2025.0.0,>=2024.4.28, but you have regex 2026.9.29 which is incompatible.


## **Step-2:- Import Libraries**

In [4]:
from PyPDF2 import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import os
import glob
import getpass

from langchain_google_genai import GoogleGenerativeAIEmbeddings
import google.generativeai as genai
from langchain_community.vectorstores import FAISS

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

C:\Users\panda\AppData\Local\Temp\ipykernel_2252\1430958021.py:9: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
C:\Users\panda\AppData\Local\Temp\ipykernel_2252\1430958021.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## **Step-3:- Set Gemini API Key (securely)**

In [5]:
api_key = os.environ.get("GOOGLE_API_KEY")

if not api_key:
    api_key = getpass.getpass("Enter your Gemini API key: ")
    os.environ["GOOGLE_API_KEY"] = api_key

genai.configure(api_key=api_key)
print("Gemini API key configured.")

Gemini API key configured.


## **Step-4:- Extract Text from PDFs**

In [6]:
def get_pdf_text(pdf_paths):
    text = ""
    for pdf_path in pdf_paths:
        pdf_reader = PdfReader(pdf_path)
        for page in pdf_reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted
    return text

## **Step-5:- Split Text into Chunks**

In [7]:
def get_text_chunks(text):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=1000)
    chunks = text_splitter.split_text(text)
    return chunks

## **Step-6:- Create Embeddings & Store in FAISS (local folder)**

In [8]:
def get_vector_store(text_chunks, index_path="faiss_index"):
    embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
    vector_store = FAISS.from_texts(text_chunks, embedding=embeddings)
    vector_store.save_local(index_path)
    print(f"FAISS vector store created successfully at ./{index_path}")

## **Step-7:- Create Conversational Chain (QA logic)**

In [9]:
def get_conversational_chain():
    prompt = ChatPromptTemplate.from_template("""
    Answer the question as detailed as possible using ONLY
    the information provided in the context.

    If the answer is not available in the context, say:
    "Answer is not available in the context."

    Do not use outside knowledge
    Do not make up information.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """)

    model = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.3
    )

    chain = prompt | model | StrOutputParser()
    return chain

## **Step-8:- Handle User Questions**

In [10]:
def user_input(user_question, index_path="faiss_index"):
    embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

    new_db = FAISS.load_local(
        index_path,
        embeddings,
        allow_dangerous_deserialization=True
    )

    docs = new_db.similarity_search(user_question, k=4)
    print(f"Retrieved {len(docs)} relevant chunks.")

    context = "\n\n".join(doc.page_content for doc in docs)

    chain = get_conversational_chain()
    res = chain.invoke({
        "context": context,
        "question": user_question
    })

    print("\nReply:")
    print(res)
    return res

## **Step-9:- Main Orchestration Function**

Put PDF files in a local folder named `pdfs/` (next to this notebook) before running this cell.
Any number of PDFs is fine

In [ ]:
def main():
    print("Chat with MultiPDF using Gemini")

    pdf_folder = "pdfs"
    index_path = "faiss_index"

    pdf_paths = sorted(glob.glob(os.path.join(pdf_folder, "*.pdf")))

    if not os.path.exists(index_path):
        if not pdf_paths:
            print(f"No PDFs found in ./{pdf_folder}/ — add some PDF files there and re-run this cell.")
            return

        print(f"\nProcessing {len(pdf_paths)} PDF(s) from ./{pdf_folder}/ ...")
        raw_text = get_pdf_text(pdf_paths)
        print(f"Extracted {len(raw_text)} characters")

        text_chunks = get_text_chunks(raw_text)
        print(f"Created {len(text_chunks)} chunks")

        get_vector_store(text_chunks, index_path)
        print("FAISS index created successfully!")
    else:
        print("Existing FAISS index found.")
        print("Using existing FAISS index...")

    while True:
        user_question = input("\nAsk a question from the PDF files (or type 'exit' to quit): ")

        if user_question.lower().strip() == "exit":
            print("Goodbye!")
            break

        if not user_question.strip():
            print("Please enter a question.")
            continue

        user_input(user_question, index_path)


if __name__ == "__main__":
    main()

Chat with MultiPDF using Gemini

Processing 2 PDF(s) from ./pdfs/ ...
Extracted 74872 characters
Created 9 chunks
FAISS vector store created successfully at ./faiss_index
FAISS index created successfully!
Retrieved 4 relevant chunks.

Reply:
The "Attention Is All You Need" paper and the "You Only Look Once" (YOLO) paper differ in their primary focus, the problem they address, and their core architectural innovations.

**"Attention Is All You Need" Paper:**
*   **Primary Focus:** This paper proposes a new simple network architecture called the Transformer, which is based solely on attention mechanisms, completely dispensing with recurrence and convolutions.
*   **Problem Addressed:** It focuses on dominant sequence transduction models, such as those used for language modeling and machine translation.
*   **Key Innovation:** The Transformer model relies entirely on an attention mechanism to draw global dependencies between input and output, allowing for significantly more parallelization